In [1]:

# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
from statsbombpy import sb

from pipeline import get_events_from_timeline, get_complete_carries, spatial_unpack

In [2]:
match_df = get_complete_carries(spatial_unpack(get_events_from_timeline(1, 3754296))).sort_values(by = 'match_seconds').reset_index(drop = True)

/home/kishor/Projects/ShotDist/shotdistributionpredictor/.venv/lib/python3.14/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [3]:
match_df = spatial_unpack(match_df, targetcolumn = 'pass_end_location', x_name = 'pass_x', y_name = 'pass_y')

In [4]:
print(match_df['type'].unique())

<StringArray>
[    'Starting XI',      'Half Start',            'Pass',   'Ball Receipt*',
           'Carry',  'Complete Carry',            'Duel',        'Pressure',
   'Ball Recovery',        'Foul Won',    'Dispossessed',      'Miscontrol',
   'Dribbled Past',           'Block',     'Goal Keeper',         'Dribble',
  'Foul Committed',       'Clearance',            'Shot',    'Interception',
        'Half End',           '50/50',    'Substitution', 'Injury Stoppage']
Length: 24, dtype: str


In [5]:
ff_events = match_df['50_50']

In [6]:
ff_normalised = pd.json_normalize(ff_events)

In [7]:
match_df[['50_50_outcome_id', '50_50_outcome']] = ff_normalised

In [9]:
ff_true = match_df[match_df['type'] == '50/50']

In [8]:
key_events = ['Ball Recovery', 'Ball Receipt*', 'Interception', 'Complete Carry', 'Offside', '50/50', 'Pass', 'Shot', 'Duel']

In [10]:
key_match_df = match_df[match_df['type'].isin(key_events)]

In [11]:
print(key_match_df['type'].unique())

<StringArray>
[          'Pass',  'Ball Receipt*', 'Complete Carry',           'Duel',
  'Ball Recovery',           'Shot',   'Interception',          '50/50']
Length: 8, dtype: str


In [12]:
key_columns = ['player', 'match_seconds', 'play_pattern', 'type','location', 'interception_outcome', 'duel_outcome', '50_50', 'carry_end_location', 'pass_end_location', 'x', 'y','end_x', 'end_y', 'pass_x', 'pass_y']

In [13]:
main_df = key_match_df[key_columns]

In [14]:
main_df = main_df.sort_values(by = 'match_seconds').reset_index(drop = True)

In [15]:
main_ff_check = main_df[main_df['type'] == '50/50']

Ball Receptions:
- Ball Receipt*
- Ball Recovery
- Duel(Won)
- 50/50(Won)
- Interception(Won)
- Pass(Recovery)
- Pass(Recovery)

Won/Recovery columns:
- duel_outcome == 'Won'
- interception_outcome == 'Won'
- 50_50_outcome == 'Won'
- pass_type == 'Recovery'
- shot_type == 'Recovery'

In [17]:
reception_types = ['Ball Receipt*', 'Ball Recovery']
open_play_phases = ['Regular Play', 'From Throw In', 'From Counter', 'From Keeper']
main_receptions = match_df[(match_df['type'].isin(reception_types)) |
                           ((match_df['type'] == 'Duel') & (match_df['duel_outcome'] == 'Won')) |
                           ((match_df['type'] == 'Interception') & (match_df['interception_outcome'] == 'Won')) |
                           ((match_df['type'] == '50_50') & (match_df['50_50_outcome'] == 'Won')) |
                           ((match_df['type'] == 'Pass') & (match_df['pass_type'] == 'Recovery')) |
                           ((match_df['type'] == 'Shot') & (match_df['shot_type'] == 'Recovery'))
]
main_receptions = main_receptions[main_receptions['play_pattern'].isin(open_play_phases)]

In [28]:
def average_y(event_df, step_length = 1, window_rad = 2.5):
    df = event_df.copy()
    df = df.sort_values(by = ['x']).reset_index(drop=True)
    x_grid = np.arange(0, 121, step=step_length)

    x_arr = df['x'].values
    y_arr = df['y'].values

    y_cumsum = np.cumsum(np.insert(y_arr, 0, 0))
    left_indices = np.searchsorted(x_arr, x_grid - window_rad, side="left")
    right_indices = np.searchsorted(x_arr, x_grid + window_rad, side="right")

    counts = right_indices - left_indices
    sums = y_cumsum[right_indices] - y_cumsum[left_indices]

    with np.errstate(divide="ignore", invalid="ignore"):
        avg_y = np.where(counts > 0, sums / counts, np.nan)

    return np.column_stack((x_grid, avg_y))

In [29]:
av_y_receptions = average_y(main_receptions)

In [30]:
def y_quad_average(event_df, step_length = 1, window_rad = 2.5):
    df = event_df.copy()
    xy_grid = average_y(df, step_length=step_length, window_rad=window_rad)

    x_vals = xy_grid[:, 0]
    y_vals = xy_grid[:, 1]
    
    fit_mask = ~np.isnan(y_vals)
    fit_x = x_vals[fit_mask]
    fit_y = y_vals[fit_mask]

    if len(fit_x) < 3:
        raise ValueError("Not enough valid window averages to fit a quadratic curve.")

    a, b, c = np.polyfit(fit_x, fit_y, deg=2)
    return [a, b, c]

In [31]:
quad_y_receptions = y_quad_average(main_receptions)

In [35]:
def get_normalised_y(event_df, step_length = 1, window_rad = 2.5):
    UPPER = 120
    LOWER = 0

    df = event_df.copy()
    coeffs = y_quad_average(df, step_length=step_length, window_rad=window_rad)

    df['curve_y'] = (coeffs[0] * (df['x']**2)) + (coeffs[1] * df['x']) + (coeffs[2])
    df["y_distance"] = df["y"] - df["curve_y"]

    above_curve = df["y"] >= df["curve_y"]
    df["boundary_distance"] = np.where(
        above_curve,
        np.abs(UPPER - df["curve_y"]),
        np.abs(df["curve_y"] - LOWER),
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        df["normalised_y"] = np.where(
            df["boundary_distance"] > 0,
            np.abs(df["y_distance"] / df["boundary_distance"]),
            0,
        )

    return df

In [36]:
normalised_receptions = get_normalised_y(main_receptions)